# Stage 4.5 v4 — TrackNet ball detector (Colab GPU training)

**Runtime → Change runtime type → GPU (T4 or A100).**

## Setup (one upload)
1. Upload **`pb_v4_upload.zip`** (built locally; code + all 5 frame caches,
   ~1.5 GB) to your Drive at `MyDrive/pb_v4_upload.zip`.
2. Run all cells. The zip is unzipped to Colab's LOCAL disk (fast); only the
   trained weights are written back to Drive.

## Split
- **Train:** pb_3min, pb_4min, pb_5min (same court, different players)
- **Val / early-stop:** pb_2min (same court)
- **Test (generalization):** pb_3min_court2 (DIFFERENT court)

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, sys
from pathlib import Path
ZIP   = '/content/drive/MyDrive/pb_v4_upload.zip'   # <- where you uploaded it
LOCAL = Path('/content/pb_v4')
if not LOCAL.exists():
    print('unzipping to local disk...')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(LOCAL)
REPO = LOCAL/'repo'; DATA = LOCAL/'data'
OUT_DRIVE = '/content/drive/MyDrive/ball_model_v4.pt'   # weights persist on Drive
sys.path.insert(0, str(REPO))
print('repo:', REPO.exists(), '| data:', DATA.exists(),
      '| clips:', sorted(p.name for p in DATA.iterdir()))

In [ ]:
try:
    import cv2  # noqa
except Exception:
    !pip -q install opencv-python-headless

In [ ]:
import logging; logging.basicConfig(level=logging.INFO)
from stages.finetune_ball_model.train_v4 import TrainConfig
from stages.finetune_ball_model._v4_cache import train_from_manifests

train_folders = [DATA/'pb_3min', DATA/'pb_4min', DATA/'pb_5min']
val_folder    = DATA/'pb_2min'
test_folder   = DATA/'pb_3min_court2'   # different court = generalization test
for f in train_folders + [val_folder, test_folder]:
    assert (f/'v4_manifest.json').exists(), f'missing manifest: {f}'
print('OK — all manifests present')

In [ ]:
cfg = TrainConfig(
    epochs=40,
    batch_size=8,        # drop to 4 on T4 if OOM; raise on A100
    lr=1e-3,
    out_path=OUT_DRIVE,
)
result = train_from_manifests(train_folders, val_folder, cfg,
                              test_folder=test_folder, num_workers=4)
print('best VAL recall (same court):', result['best_val_recall'])
print('TEST recall (different court, generalization):', result['test_recall'])

## Reading the result (acceptance bar: test recall ≥ 0.80)
- **val recall** (same court, new players) — the easy case; should be high.
- **test recall** (different court) — the number that matters.
  - **Both high** → generalizes across courts. Done.
  - **Val high, test low** → overfit to the training court → add
    pb_3min_court2 into TRAINING + grab a new court as the next test
    (per-venue fine-tune loop).
  - **Both low** → bump input to 1080p (set PROC_H,PROC_W=1088,1920 in
    `_v4_data.py`), re-run prepare_v4 locally, re-upload, retrain.
- Download `MyDrive/ball_model_v4.pt` to `data/models/` locally; report
  both recalls.

In [ ]:
import json
rep = json.load(open('/content/drive/MyDrive/validation_report.json'))
print('val', rep['val_clip'], rep['best_val_recall'],
      '| test', rep['test_clip'], rep['test_recall_at_best_val'])
for h in rep['history'][-6:]: print(h)